<a href="https://colab.research.google.com/github/Abdu1l-0/AIDC-W3-LABS/blob/W3D4/W3D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Cell 1:
import subprocess
import os
# 1. Install python3.10 and venv support on Colab
subprocess.run(["sudo", "apt-get", "update", "-y"], check=True)
subprocess.run(["sudo", "apt-get", "install", "python3.10", "python3.10-venv", "python3.10-dev", "-y"], check=True)
# 2. Create the virtual environment
subprocess.run(["python3.10", "-m", "venv", "/content/venv"], check=True)
# 3. Upgrade pip inside the venv
subprocess.run(["/content/venv/bin/python", "-m", "pip", "install", "--upgrade", "pip"], check=True)
# 4. Install the serving pins + autoawq for Day 4
subprocess.run([
    "/content/venv/bin/python", "-m", "pip", "install", "-q",
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "autoawq==0.2.*",
    "httpx==0.27.*",
    "openai==1.54.*"
], check=True)
print("Python 3.10 venv ready with AWQ serving pins installed!")

Python 3.10 venv ready with AWQ serving pins installed!


In [2]:
import subprocess
import os
import time
import urllib.request
import urllib.error
SERVER_LOG = "/content/server.log"
PORT = 8000
SERVER_ARGS = [
    "/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype", "half",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--port", str(PORT),
    "--quantization", "awq",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes"
]
print("Launching AWQ server...")
logf = open(SERVER_LOG, "wb")
server_proc = subprocess.Popen(
    SERVER_ARGS, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True
)
print(f"Server PID {server_proc.pid}, logging to {SERVER_LOG}")

Launching AWQ server...
Server PID 35703, logging to /content/server.log


In [4]:
url = f"http://localhost:{PORT}/v1/models"
timeout_s = 300
deadline = time.time() + timeout_s
print("Waiting for server health check...")
while time.time() < deadline:
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            if r.status == 200:
                print(f"Server healthy! Endpoint live at {url}")
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass
    time.sleep(3)
else:
    print("Timed out waiting for server. Last 30 lines of /content/server.log:")
    with open(SERVER_LOG, "r", errors="replace") as fh:
        print("".join(fh.readlines()[-30:]))

# The server was already launched in the previous cell via subprocess.Popen,
# so we do not need to call launch_server(SERVER_ARGS) here.

Waiting for server health check...
Server healthy! Endpoint live at http://localhost:8000/v1/models


In [5]:
!nvidia-smi --query-gpu=memory.used --format=csv,noheader
# note the number; compare to yesterday's fp16 resident VRAM

11723 MiB


In [6]:
SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        messages=[{"role": "user", "content": p}], max_tokens=200)
    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content, "\n")

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server is responsible for processing incoming requests and generating responses based on the input data, typically for tasks such as machine learning model inference, predictive analytics, and automated decision-making. 

PROMPT: A user asks for the weather in Riyadh and the time ...
To fetch the weather in Riyadh and the time in Tokyo for a specific date, you would typically make two API calls. Assuming the services that provide these information are accessible through two different APIs (API One for weather and API Two for time), you would issue the following calls:

For weather in Riyadh:
```plaintext
http://api.weather.com/w/chat?apiKey=[your_api_key]&conditions=all&urls=http://weather.com/re/ (replaced "re" with Riyadh)
```

To fetch the time in Tokyo (`Tokyo`):
```plaintext
http://api.timeZone.com/timetables/v1/timeships.json?key=apikey&cityIds=891&api-language=ru (Japan time zone ID 891)
```

Replace `[y

In [7]:
from openai import OpenAI

# Two tools the model may call. Shapes match the OpenAI tools schema.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate an arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "e.g. 23 * 19"},
                },
                "required": ["expression"],
            },
        },
    },
]

# The 3 canonical prompts. Each carries how many attempts (k) it gets and how many
# tool calls a correct answer makes. n = sum of k = 10.
#   two_tool:   needs BOTH tools (weather + calculator)   -> expect >= 1 call
#   single:     needs ONE tool                            -> expect >= 1 call
#   distractor: needs NO tool, must NOT call one          -> expect 0 calls
CANONICAL = [
    {
        "id": "two_tool",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Riyadh, and what is 23 multiplied "
                  "by 19? Use your tools.",
    },
    {
        "id": "single",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Tokyo right now? Use your tools.",
    },
    {
        "id": "distractor",
        "k": 2,
        "wants_call": False,
        "prompt": "In one sentence, explain what a tool call is. Do not call "
                  "any tool; just answer.",
    },
]


def _tool_calls_of(message) -> list:
    """Return the parsed tool_calls list on a response message, or []."""
    tc = getattr(message, "tool_calls", None)
    return list(tc) if tc else []


def _valid_call(call) -> bool:
    """A tool call is valid if it names a known function and its arguments
    parse as JSON with the required field present."""
    import json
    try:
        fn = call.function.name
        if fn not in ("get_weather", "calculate"):
            return False
        args = json.loads(call.function.arguments or "{}")
    except (AttributeError, ValueError):
        return False
    if fn == "get_weather":
        return isinstance(args.get("city"), str) and bool(args["city"])
    if fn == "calculate":
        return isinstance(args.get("expression"), str) and bool(args["expression"])
    return False


def run_smoke(base_url: str, model: str, temperature: float = 0.0) -> dict:
    """Run the smoke test. Returns a result dict with counts and the pass gate."""
    client = OpenAI(base_url=base_url, api_key="not-needed")

    total_attempts = 0
    valid_call_attempts = 0          # attempts that returned >=1 valid tool call
    distractor_attempts = 0
    distractor_call_free = 0         # distractor attempts that made NO tool call
    per_prompt = {}

    for spec in CANONICAL:
        pid, k, wants = spec["id"], spec["k"], spec["wants_call"]
        got_valid = 0
        got_call_free = 0
        for _ in range(k):
            total_attempts += 1
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": spec["prompt"]}],
                tools=TOOLS,
                tool_choice="auto",
                temperature=temperature,
                max_tokens=256,
            )
            msg = resp.choices[0].message
            calls = _tool_calls_of(msg)
            any_valid = any(_valid_call(c) for c in calls)

            if wants:
                # a "wants a call" prompt counts toward the 8/10 gate when it
                # returns at least one valid tool call
                if any_valid:
                    valid_call_attempts += 1
                    got_valid += 1
            else:
                # the distractor counts toward the 8/10 gate when it correctly
                # makes NO tool call, and separately toward distractor compliance
                distractor_attempts += 1
                if not calls:
                    valid_call_attempts += 1
                    distractor_call_free += 1
                    got_call_free += 1

        per_prompt[pid] = {"k": k, "wants_call": wants,
                           "valid": got_valid, "call_free": got_call_free}

    # gate: >=8/10 correct behaviours AND distractor call-free in the majority
    distractor_majority = (distractor_call_free * 2 > distractor_attempts) \
        if distractor_attempts else True
    passed = (valid_call_attempts >= 8) and distractor_majority

    return {
        "model": model,
        "total_attempts": total_attempts,       # 10
        "score": valid_call_attempts,           # correct behaviours, out of 10
        "distractor_attempts": distractor_attempts,
        "distractor_call_free": distractor_call_free,
        "distractor_majority_clean": distractor_majority,
        "per_prompt": per_prompt,
        "passed": passed,
    }
result = run_smoke(base_url="http://localhost:8000/v1",
                   model="Qwen/Qwen2.5-1.5B-Instruct-AWQ")
print(result)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [8]:
import json
with open("smoke_result.json", "w") as f:
    json.dump(result, f, indent=2)

In [12]:
# Green-check verifier for Lab W3D4 (quantise and lock).
# Paste this as the last cell of your day-4 notebook and run it. It reads
# smoke_result.json (written from the smoke test) and model-lock.md, and checks
# that the smoke score meets the gate and that the lock file is fully filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) smoke result
    if not os.path.exists("smoke_result.json"):
        fail("smoke_result.json not found; write it in Cell 5")
    try:
        with open("smoke_result.json") as fh:
            result = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"smoke_result.json is not valid JSON: {exc}")

    for key in ("score", "total_attempts", "distractor_majority_clean", "passed"):
        if key not in result:
            fail(f"smoke_result.json missing key: {key}")

    score = result["score"]
    total = result["total_attempts"]
    if not isinstance(score, int) or not isinstance(total, int):
        fail("score and total_attempts must be integers")
    if total != 10:
        fail(f"total_attempts is {total}, the smoke test defines n=10")
    if score < 8:
        fail(f"smoke score {score}/10 is below the 8/10 gate")
    if not result["distractor_majority_clean"]:
        fail("distractor did not stay call-free in the majority; a model that "
             "always calls a tool fails the real consumer")
    if not result["passed"]:
        fail("smoke test reports passed=false")

    # 2) model-lock.md fully filled in
    if not os.path.exists("model-lock.md"):
        fail("model-lock.md not found")
    with open("model-lock.md") as fh:
        lock = fh.read()
    remaining = re.findall(r"FILL:", lock)
    if remaining:
        fail(f"model-lock.md has {len(remaining)} unfilled FILL: placeholders")
    # require a concrete model id line
    if not re.search(r"Model id:\s*\S+", lock):
        fail("model-lock.md has no concrete Model id")

    print(f"smoke score: {score}/{total}, distractor clean: "
          f"{result['distractor_majority_clean']}")
    print("model-lock.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


smoke score: 10/10, distractor clean: True
model-lock.md: all fields filled
GREEN CHECK: PASS
